1. Set JSON Parsing functions

In [14]:
import json
import ssl
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen


def get_ssl_context(verify_ssl=True):
    """Return an SSL context based on verification preference."""
    return ssl.create_default_context() if verify_ssl else ssl._create_unverified_context()


def fetch_json_from_api(api_url, headers=None, timeout=30, verify_ssl=True):
    """Fetch JSON data from a configurable API endpoint."""
    headers = headers or {"User-Agent": "python-sejm-api-client/1.0"}
    request = Request(api_url, headers=headers)
    ssl_context = get_ssl_context(verify_ssl)

    try:
        with urlopen(request, timeout=timeout, context=ssl_context) as response:
            body = response.read()
            return json.loads(body.decode("utf-8"))
    except HTTPError as exc:
        raise RuntimeError(f"API request failed: {exc.code} {exc.reason}") from exc
    except URLError as exc:
        if isinstance(exc.reason, ssl.SSLError):
            if not verify_ssl:
                raise RuntimeError(
                    "SSL verification failed even though verify_ssl is disabled. "
                    "Check if outbound HTTPS traffic is allowed."
                ) from exc
            raise RuntimeError(
                "SSL error: certificate verification failed. "
                "If your environment uses a self-signed proxy or custom CA, disable verification "
                "with verify_ssl=False or install the correct CA certificate."
            ) from exc
        raise RuntimeError(f"API network error: {exc}") from exc
    except ssl.SSLError as exc:
        if not verify_ssl:
            raise RuntimeError(
                "SSL verification failed even though verify_ssl is disabled. "
                "Check if outbound HTTPS traffic is allowed."
            ) from exc
        raise RuntimeError(
            "SSL error: certificate verification failed. "
            "If your environment uses a self-signed proxy or custom CA, disable verification "
            "with verify_ssl=False or install the correct CA certificate."
        ) from exc
    except ValueError as exc:
        raise RuntimeError("Failed to parse JSON response from API") from exc


def save_payload(payload, path):
    """Save the retrieved payload to a JSON file."""
    output_path = Path(path)
    output_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    return output_path


def main(api_url, output_path=None, verify_ssl=True):
    payload = fetch_json_from_api(api_url, verify_ssl=verify_ssl)
    count = len(payload) if isinstance(payload, (list, tuple)) else None

    print("API payload loaded.")
    print(f"Payload type: {type(payload).__name__}")
    if count is not None:
        print(f"Item count: {count}")

    if output_path:
        saved_path = save_payload(payload, output_path)
        print(f"Saved JSON payload to: {saved_path}")

    return payload


# Example usage:

api_url = "https://api.sejm.gov.pl/sejm/term8/MP"
output_file = Path.cwd() / "sejm_term.json"
payload = main(api_url, output_path=output_file, verify_ssl=False)

API payload loaded.
Payload type: list
Item count: 505
Saved JSON payload to: /Users/marek/Repos/marumarecki_ingestion/sejm_api_pipeline/sejm_term.json


In [15]:
def load_payload_to_R2(path):
    """Load the payload from file."""
    # Import python packages
    from dataplane import s3_upload
    import os
    import boto3
    from botocore.client import Config
    from dotenv import load_dotenv
    from importlib.resources import path
    import json


    load_dotenv()


    # 1. Account ID
    AccountID = os.environ["secret_dp_S3_ACCOUNT_ID"]

    # 2. Bucket name
    Bucket = os.environ["secret_dp_BUCKET_NAME"]

    # 3. Client access key
    ClientAccessKey = os.environ["secret_dp_S3_ACCESS_KEY"]

    # 4. Client secret
    ClientSecret = os.environ["secret_dp_S3_SECRET"]

    # 5. Connection url
    ConnectionUrl = f"https://{AccountID}.r2.cloudflarestorage.com"

    # Create a client to connect to Cloudflare's R2 Storage
    S3Connect = boto3.client(
        's3',
        endpoint_url=ConnectionUrl,
        aws_access_key_id=ClientAccessKey,
        aws_secret_access_key=ClientSecret,
        config=Config(signature_version='s3v4'),
        region_name='us-east-1'

    )

    # Upload to R2 using S3 compatible API
    rs = s3_upload(Bucket=Bucket, 
        S3Client=S3Connect,
        TargetFilePath=f"test/my file.json",
        UploadObject=UploadObject,
        UploadMethod="Object"
        )
    return(rs)
    